# 🔥 FictiPay Datathon — Full Churn Prediction Pipeline (Colab Edition)

**Pipeline**: Feature Engineering → Model Zoo Training (10-Fold CV) → Rank-Average Ensemble → Submission

**Features**: 234 engineered features including graph/network topology, per-type monetary decay, temporal behavior, and balance-transaction interactions.

**Models**: LightGBM, XGBoost, CatBoost, Random Forest, Logistic Regression, MLP — all ensembled with Optuna-optimized power-rank blending.

> ⚠️ **Run Cell 2 FIRST** to prevent Colab from disconnecting during long training runs.

## 🛡️ Step 0: Anti-Idle Protection
Run this cell to prevent Google Colab from disconnecting due to inactivity. It injects JavaScript that clicks the page every 60 seconds.

In [ ]:
# === ANTI-IDLE: Prevents Colab from disconnecting ===
# This cell injects JavaScript that simulates activity every 60 seconds.
# It also starts a background Python thread that prints a heartbeat every 5 minutes.

import threading, time, IPython

# JavaScript anti-idle (clicks connect button periodically)
js_code = """
function KeepColabAlive() {
    console.log("KeepAlive: pinging at " + new Date().toLocaleTimeString());
    // Click the "connect" button if it appears (reconnect on disconnect)
    var buttons = document.querySelectorAll("colab-connect-button");
    if (buttons.length > 0) { buttons[0].click(); }
    // Also trigger a minor DOM event to reset idle timer
    document.dispatchEvent(new KeyboardEvent('keydown', {key: 'Shift'}));
}
setInterval(KeepColabAlive, 60000);  // Every 60 seconds
console.log("✅ KeepAlive installed — Colab will not disconnect due to idle.");
"""
IPython.display.display(IPython.display.Javascript(js_code))

# Python background heartbeat thread
def _heartbeat():
    while True:
        time.sleep(300)  # Every 5 minutes
        print(f"💓 Heartbeat: {time.strftime('%H:%M:%S')} — session alive", flush=True)

_hb_thread = threading.Thread(target=_heartbeat, daemon=True)
_hb_thread.start()
print("✅ Anti-idle protection active (JS click every 60s + Python heartbeat every 5min)")

## 📦 Step 1: Install Dependencies

In [ ]:
!pip install -q lightgbm xgboost catboost optuna shap
import importlib
for pkg in ["lightgbm", "xgboost", "catboost", "optuna", "shap"]:
    try:
        importlib.import_module(pkg)
        print(f"  ✅ {pkg}")
    except ImportError:
        print(f"  ❌ {pkg} — install failed")

## 📂 Step 2: Mount Google Drive & Set Dataset Path

Upload your `bkash-presents-nsucec-datathon` folder to Google Drive, then update the path below.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# === SET YOUR DATASET PATH HERE ===
# Point this to wherever you uploaded the dataset in Google Drive
base_path = "/content/drive/MyDrive/bkash-presents-nsucec-datathon/public"

import os
assert os.path.exists(base_path), f"Dataset path not found: {base_path}"
print("✅ Dataset found at:", base_path)
print("Files:", os.listdir(base_path))

## ⚙️ Step 3: Imports & Configuration

In [ ]:
import os, gc, glob, time, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, f1_score, recall_score, precision_score, brier_score_loss, roc_curve
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.isotonic import IsotonicRegression
from scipy.stats import rankdata
import lightgbm as lgb
import xgboost as xgb

try:
    from catboost import CatBoostClassifier
    CATBOOST_AVAILABLE = True
    print("✅ CatBoost available")
except ImportError:
    CATBOOST_AVAILABLE = False
    print("⚠️ CatBoost not available — will be skipped")

try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    OPTUNA_AVAILABLE = True
    print("✅ Optuna available")
except ImportError:
    OPTUNA_AVAILABLE = False
    print("⚠️ Optuna not available — will use hill-climbing fallback")

warnings.filterwarnings("ignore")

# Pipeline constants
N_SPLITS = 10
RANDOM_STATE = 42
print(f"\n📊 Pipeline: {N_SPLITS}-Fold Stratified CV, seed={RANDOM_STATE}")

## 🧪 Step 4: Feature Engineering

This section defines all feature engineering functions and runs them. **234 features** are extracted covering:
- Demographics & tenure
- Transaction counts, sums, per-type breakdowns (Jan/Feb/March)
- **Graph/Network features** (unique counterparties, concentration, HHI)
- **Temporal behavior** (active days, hour-of-day patterns)
- **Per-type spend decay** (MerchantPay, P2P, BillPay trends)
- **Balance trends** (micro-windows, slopes, zero-balance days)
- **Balance × Transaction interactions** (spend ratio, runway, intensity)

In [ ]:
import os
import gc
import glob
import pandas as pd
import numpy as np

# base_path is set in Step 2 above

def load_kyc(target_ids):
    """Load and preprocess KYC metadata for target customers."""
    print("Loading and preprocessing KYC...")
    kyc = pd.read_parquet(os.path.join(base_path, "kyc.parquet"))
    kyc = kyc[kyc["ACCOUNT_ID"].isin(target_ids)].copy()
    
    ref_date = pd.to_datetime("2024-03-31")
    kyc["tenure_days"] = (ref_date - pd.to_datetime(kyc["ACCOUNT_OPEN_DATE"])).dt.days
    kyc["GENDER"] = kyc["GENDER"].fillna("Unknown")
    kyc["REGION"] = kyc["REGION"].fillna("Unknown")
    kyc = pd.get_dummies(kyc, columns=["GENDER", "REGION"], prefix=["gender", "region"], dtype=float)
    kyc = kyc.drop(columns=["ACCOUNT_TYPE", "ACCOUNT_OPEN_DATE"])
    kyc = kyc.set_index("ACCOUNT_ID")
    return kyc


def compute_recency(target_ids):
    """Separate lightweight pass to compute days_since_last_trx.
    Only loads SRC_ACCOUNT, DST_ACCOUNT, TRX_DATETIME — no TRX_AMT."""
    print("Computing recency (lightweight datetime pass)...")
    trx_files = sorted(glob.glob(os.path.join(base_path, "transactions", "*.parquet")))
    
    last_trx = {}  # ACCOUNT_ID -> max datetime
    
    for filepath in trx_files:
        fname = os.path.basename(filepath)
        print(f"  Scanning {fname} for recency...")
        
        # Only load the two ID columns and datetime — skip TRX_AMT and TRX_TYPE
        df = pd.read_parquet(filepath, columns=["SRC_ACCOUNT", "DST_ACCOUNT", "TRX_DATETIME"])
        df["TRX_DATETIME"] = pd.to_datetime(df["TRX_DATETIME"])
        
        # Source side
        mask_src = df["SRC_ACCOUNT"].isin(target_ids)
        src_last = df.loc[mask_src].groupby("SRC_ACCOUNT")["TRX_DATETIME"].max()
        for acct, dt in src_last.items():
            if acct not in last_trx or dt > last_trx[acct]:
                last_trx[acct] = dt
        del src_last, mask_src
        
        # Destination side
        mask_dst = df["DST_ACCOUNT"].isin(target_ids)
        dst_last = df.loc[mask_dst].groupby("DST_ACCOUNT")["TRX_DATETIME"].max()
        for acct, dt in dst_last.items():
            if acct not in last_trx or dt > last_trx[acct]:
                last_trx[acct] = dt
        del dst_last, mask_dst, df
        gc.collect()
    
    ref_date = pd.to_datetime("2024-03-31")
    recency_series = pd.Series(last_trx)
    recency_series.index.name = "ACCOUNT_ID"
    recency_df = pd.DataFrame({"days_since_last_trx": (ref_date - recency_series).dt.days})
    
    print(f"  Recency computed for {len(recency_df):,} customers.")
    del last_trx, recency_series
    gc.collect()
    return recency_df


def compute_advanced_march_features(target_ids):
    """Load March transactions and extract highly predictive recency, velocity, and micro-window features.
    This runs fast because it is limited to a single month (March)."""
    print("Computing advanced March transaction features...")
    filepath = os.path.join(base_path, "transactions", "trx_2024-03.parquet")
    if not os.path.exists(filepath):
        print("  March transaction file not found! Skipping advanced March features.")
        return pd.DataFrame(index=list(target_ids))
        
    df = pd.read_parquet(filepath, columns=["SRC_ACCOUNT", "DST_ACCOUNT", "TRX_DATETIME", "TRX_TYPE", "TRX_AMT"])
    df["TRX_DATETIME"] = pd.to_datetime(df["TRX_DATETIME"])
    
    # Filter for target customer IDs
    df_src = df[df["SRC_ACCOUNT"].isin(target_ids)].copy()
    df_dst = df[df["DST_ACCOUNT"].isin(target_ids)].copy()
    
    ref_date = pd.to_datetime("2024-03-31 23:59:59")
    march_adv = pd.DataFrame(index=list(target_ids))
    march_adv.index.name = "ACCOUNT_ID"
    
    # 1. Outbound and Inbound Recencies
    print("  Calculating directional recencies...")
    out_last = df_src.groupby("SRC_ACCOUNT")["TRX_DATETIME"].max()
    in_last = df_dst.groupby("DST_ACCOUNT")["TRX_DATETIME"].max()
    
    march_adv["days_since_last_outbound"] = (ref_date - out_last).dt.days
    march_adv["days_since_last_inbound"] = (ref_date - in_last).dt.days
    
    # 2. Recency by transaction type (outbound and inbound)
    print("  Calculating recency by transaction types...")
    trx_types = ["P2P", "MerchantPay", "BillPay", "CashIn", "CashOut"]
    for ttype in trx_types:
        # Outbound types: P2P, MerchantPay, BillPay, CashOut (initiated by customer)
        if ttype in ["P2P", "MerchantPay", "BillPay", "CashOut"]:
            last_dt = df_src[df_src["TRX_TYPE"] == ttype].groupby("SRC_ACCOUNT")["TRX_DATETIME"].max()
            march_adv[f"days_since_last_{ttype}"] = (ref_date - last_dt).dt.days
        # Inbound types: CashIn, P2P (received by customer)
        if ttype in ["CashIn", "P2P"]:
            last_dt = df_dst[df_dst["TRX_TYPE"] == ttype].groupby("DST_ACCOUNT")["TRX_DATETIME"].max()
            prefix = "received" if ttype == "P2P" else "last"
            march_adv[f"days_since_{prefix}_{ttype}"] = (ref_date - last_dt).dt.days
            
    # 3. Micro-window counts and sums (Last 7 days: March 25-31, Last 14 days: March 18-31)
    print("  Calculating micro-window statistics (7d and 14d)...")
    date_7d = pd.to_datetime("2024-03-25 00:00:00")
    date_14d = pd.to_datetime("2024-03-18 00:00:00")
    
    # Outbound 7d/14d
    df_src_7d = df_src[df_src["TRX_DATETIME"] >= date_7d]
    df_src_14d = df_src[df_src["TRX_DATETIME"] >= date_14d]
    
    src_7d_agg = df_src_7d.groupby("SRC_ACCOUNT")["TRX_AMT"].agg(
        out_count_last_7d="count",
        out_sum_last_7d="sum"
    )
    src_14d_agg = df_src_14d.groupby("SRC_ACCOUNT")["TRX_AMT"].agg(
        out_count_last_14d="count",
        out_sum_last_14d="sum"
    )
    
    march_adv = march_adv.join(src_7d_agg, how="left").join(src_14d_agg, how="left")
    
    # Inbound 7d/14d
    df_dst_7d = df_dst[df_dst["TRX_DATETIME"] >= date_7d]
    df_dst_14d = df_dst[df_dst["TRX_DATETIME"] >= date_14d]
    
    dst_7d_agg = df_dst_7d.groupby("DST_ACCOUNT")["TRX_AMT"].agg(
        in_count_last_7d="count",
        in_sum_last_7d="sum"
    )
    dst_14d_agg = df_dst_14d.groupby("DST_ACCOUNT")["TRX_AMT"].agg(
        in_count_last_14d="count",
        in_sum_last_14d="sum"
    )
    
    march_adv = march_adv.join(dst_7d_agg, how="left").join(dst_14d_agg, how="left").fillna(0)
    
    # 4. Outbound transaction velocity in March
    # We will load the total March count and sum to calculate velocity ratios
    march_total_src = df_src.groupby("SRC_ACCOUNT")["TRX_AMT"].agg(
        out_count_march="count",
        out_sum_march="sum"
    )
    
    march_adv = march_adv.join(march_total_src, how="left").fillna(0)
    
    march_adv["out_count_velocity_7d"] = march_adv["out_count_last_7d"] / (march_adv["out_count_march"] + 1e-5)
    march_adv["out_count_velocity_14d"] = march_adv["out_count_last_14d"] / (march_adv["out_count_march"] + 1e-5)
    march_adv["out_sum_velocity_7d"] = march_adv["out_sum_last_7d"] / (march_adv["out_sum_march"] + 1e-5)
    march_adv["out_sum_velocity_14d"] = march_adv["out_sum_last_14d"] / (march_adv["out_sum_march"] + 1e-5)
    
    # Drop columns that are duplicated in the main aggregation
    march_adv = march_adv.drop(columns=["out_count_march", "out_sum_march"])
    
    # Fill missing recency values with 31 days (max for March)
    recency_cols = [c for c in march_adv.columns if c.startswith("days_since_") or c.startswith("days_since_last_")]
    for col in recency_cols:
        march_adv[col] = march_adv[col].fillna(31)
        
    print(f"  Advanced March features computed: {march_adv.shape[1]} features.")
    del df, df_src, df_dst, out_last, in_last, df_src_7d, df_src_14d, src_7d_agg, src_14d_agg
    del df_dst_7d, df_dst_14d, dst_7d_agg, dst_14d_agg, march_total_src
    gc.collect()
    
    return march_adv


def aggregate_transactions_one_month(filepath, month, target_ids, kyc_type_map=None):
    """Process a SINGLE month of transactions with comprehensive feature extraction.
    Loads TRX_DATETIME for temporal features (active days, hour patterns)."""
    print(f"  Aggregating {month} transactions ({os.path.basename(filepath)})...")
    
    # Load ALL columns for full feature extraction
    df = pd.read_parquet(filepath, columns=["SRC_ACCOUNT", "DST_ACCOUNT", "TRX_TYPE", "TRX_AMT", "TRX_DATETIME"])
    df["TRX_DATETIME"] = pd.to_datetime(df["TRX_DATETIME"])
    
    # ---- Outbound (SRC) ----
    df_src = df[df["SRC_ACCOUNT"].isin(target_ids)].copy()
    
    out_grp = df_src.groupby("SRC_ACCOUNT")
    out_agg = out_grp["TRX_AMT"].agg(
        out_count="count",
        out_sum="sum",
        out_avg="mean",
        out_std="std",
        out_max="max",
        out_median="median"
    )
    out_agg.index.name = "ACCOUNT_ID"
    out_agg["out_std"] = out_agg["out_std"].fillna(0)
    
    # Per-type counts (existing)
    type_counts = df_src.groupby(["SRC_ACCOUNT", "TRX_TYPE"]).size().unstack(fill_value=0)
    type_counts.index.name = "ACCOUNT_ID"
    type_counts.columns = [f"count_{col}_{month}" for col in type_counts.columns]
    
    # === NEW: Per-type monetary SUMS ===
    type_sums = df_src.groupby(["SRC_ACCOUNT", "TRX_TYPE"])["TRX_AMT"].sum().unstack(fill_value=0)
    type_sums.index.name = "ACCOUNT_ID"
    type_sums.columns = [f"sum_{col}_{month}" for col in type_sums.columns]
    
    # === NEW: Graph/Network features (outbound) ===
    # Unique destination counterparties
    unique_dst = df_src.groupby("SRC_ACCOUNT")["DST_ACCOUNT"].nunique()
    unique_dst.index.name = "ACCOUNT_ID"
    unique_dst.name = f"unique_dst_{month}"
    
    # Unique destinations by type (merchant, biller, P2P)
    if kyc_type_map is not None:
        df_src["DST_TYPE"] = df_src["DST_ACCOUNT"].map(kyc_type_map).fillna("Unknown")
        for acct_type in ["Merchant", "Biller"]:
            type_mask = df_src["DST_TYPE"] == acct_type
            type_unique = df_src[type_mask].groupby("SRC_ACCOUNT")["DST_ACCOUNT"].nunique()
            type_unique.index.name = "ACCOUNT_ID"
            type_unique.name = f"unique_{acct_type.lower()}_{month}"
            out_agg = out_agg.join(type_unique, how="left")
        # P2P unique recipients (Customer-to-Customer)
        p2p_mask = df_src["TRX_TYPE"] == "P2P"
        p2p_unique = df_src[p2p_mask].groupby("SRC_ACCOUNT")["DST_ACCOUNT"].nunique()
        p2p_unique.index.name = "ACCOUNT_ID"
        p2p_unique.name = f"unique_p2p_dst_{month}"
        out_agg = out_agg.join(p2p_unique, how="left")
    
    # Counterparty concentration: top-1 destination share of total volume
    dst_vol = df_src.groupby(["SRC_ACCOUNT", "DST_ACCOUNT"])["TRX_AMT"].sum()
    total_vol = dst_vol.groupby("SRC_ACCOUNT").sum()
    top1_vol = dst_vol.groupby("SRC_ACCOUNT").max()
    top1_share = (top1_vol / (total_vol + 1e-5))
    top1_share.index.name = "ACCOUNT_ID"
    top1_share.name = f"top1_dst_share_{month}"
    out_agg = out_agg.join(top1_share, how="left")
    
    # HHI (Herfindahl-Hirschman Index) of destination concentration
    dst_shares = dst_vol.groupby("SRC_ACCOUNT").apply(
        lambda x: ((x / (x.sum() + 1e-5)) ** 2).sum()
    )
    dst_shares.index.name = "ACCOUNT_ID"
    dst_shares.name = f"hhi_dst_{month}"
    out_agg = out_agg.join(dst_shares, how="left")
    del dst_vol, total_vol, top1_vol, top1_share, dst_shares
    
    # === NEW: Temporal features (outbound) ===
    df_src["hour"] = df_src["TRX_DATETIME"].dt.hour
    df_src["date"] = df_src["TRX_DATETIME"].dt.date
    
    # Active days (distinct dates with transactions)
    active_days = df_src.groupby("SRC_ACCOUNT")["date"].nunique()
    active_days.index.name = "ACCOUNT_ID"
    active_days.name = f"active_days_{month}"
    out_agg = out_agg.join(active_days, how="left")
    
    # Hour-of-day std (behavioral regularity)
    hour_std = df_src.groupby("SRC_ACCOUNT")["hour"].std().fillna(0)
    hour_std.index.name = "ACCOUNT_ID"
    hour_std.name = f"trx_hour_std_{month}"
    out_agg = out_agg.join(hour_std, how="left")
    
    # % of transactions in business hours (9am-5pm)
    df_src["is_business"] = df_src["hour"].between(9, 17).astype(int)
    biz_pct = df_src.groupby("SRC_ACCOUNT")["is_business"].mean()
    biz_pct.index.name = "ACCOUNT_ID"
    biz_pct.name = f"pct_business_hours_{month}"
    out_agg = out_agg.join(biz_pct, how="left")
    
    del df_src
    gc.collect()
    
    # ---- Inbound (DST) ----
    df_dst = df[df["DST_ACCOUNT"].isin(target_ids)].copy()
    
    # === ENRICHED: Full inbound stats ===
    in_agg = df_dst.groupby("DST_ACCOUNT")["TRX_AMT"].agg(
        in_count="count",
        in_sum="sum",
        in_avg="mean",
        in_std="std",
        in_max="max"
    )
    in_agg.index.name = "ACCOUNT_ID"
    in_agg["in_std"] = in_agg["in_std"].fillna(0)
    
    # Unique source counterparties (in-degree)
    unique_src = df_dst.groupby("DST_ACCOUNT")["SRC_ACCOUNT"].nunique()
    unique_src.index.name = "ACCOUNT_ID"
    unique_src.name = f"unique_src_{month}"
    in_agg = in_agg.join(unique_src, how="left")
    
    # Inbound per-type counts
    in_type_counts = df_dst.groupby(["DST_ACCOUNT", "TRX_TYPE"]).size().unstack(fill_value=0)
    in_type_counts.index.name = "ACCOUNT_ID"
    in_type_counts.columns = [f"in_count_{col}_{month}" for col in in_type_counts.columns]
    
    # Inbound active days
    df_dst["date"] = pd.to_datetime(df_dst["TRX_DATETIME"]).dt.date
    in_active_days = df_dst.groupby("DST_ACCOUNT")["date"].nunique()
    in_active_days.index.name = "ACCOUNT_ID"
    in_active_days.name = f"in_active_days_{month}"
    in_agg = in_agg.join(in_active_days, how="left")
    
    del df_dst, df
    gc.collect()
    
    # ---- Merge into single month DataFrame ----
    month_agg = pd.DataFrame(index=list(target_ids))
    month_agg.index.name = "ACCOUNT_ID"
    month_agg = month_agg.join(out_agg, how="left").fillna(0)
    month_agg = month_agg.join(type_counts, how="left").fillna(0)
    month_agg = month_agg.join(type_sums, how="left").fillna(0)
    month_agg = month_agg.join(unique_dst, how="left").fillna(0)
    month_agg = month_agg.join(in_agg, how="left").fillna(0)
    month_agg = month_agg.join(in_type_counts, how="left").fillna(0)
    
    # === NEW: In/Out ratio ===
    month_agg["in_out_ratio"] = month_agg["in_count"] / (month_agg["out_count"] + 1)
    
    del out_agg, type_counts, type_sums, unique_dst, in_agg, in_type_counts
    gc.collect()
    
    # Rename columns with month suffix (skip already-suffixed columns)
    new_cols = []
    for col in month_agg.columns:
        already_suffixed = False
        for sfx in [f"_{month}", f"_{month}"]:
            if col.endswith(sfx):
                already_suffixed = True
                break
        if already_suffixed:
            new_cols.append(col)
        else:
            new_cols.append(f"{col}_{month}")
    month_agg.columns = new_cols
    
    return month_agg


def aggregate_transactions(target_ids, kyc_type_map=None):
    """Aggregate transactions across all available months, one at a time."""
    print("Aggregating transactions (memory-safe, one month at a time)...")
    trx_files = sorted(glob.glob(os.path.join(base_path, "transactions", "*.parquet")))
    months = ["jan", "feb", "march"]
    
    trx_agg = None
    
    for month_idx, filepath in enumerate(trx_files):
        if month_idx >= len(months):
            break
        month = months[month_idx]
        
        month_df = aggregate_transactions_one_month(filepath, month, target_ids, kyc_type_map)
        
        if trx_agg is None:
            trx_agg = month_df
        else:
            trx_agg = trx_agg.join(month_df, how="outer").fillna(0)
            del month_df
            gc.collect()
    
    # ---- Cross-Month Features ----
    available_months = months[:min(len(trx_files), len(months))]
    
    # Totals
    trx_agg["out_count_total"] = sum(trx_agg.get(f"out_count_{m}", 0) for m in available_months)
    trx_agg["out_sum_total"] = sum(trx_agg.get(f"out_sum_{m}", 0) for m in available_months)
    trx_agg["in_count_total"] = sum(trx_agg.get(f"in_count_{m}", 0) for m in available_months)
    trx_agg["in_sum_total"] = sum(trx_agg.get(f"in_sum_{m}", 0) for m in available_months)
    
    # Total active days across all months
    trx_agg["active_days_total"] = sum(trx_agg.get(f"active_days_{m}", 0) for m in available_months)
    
    # Total unique destinations across all months
    trx_agg["unique_dst_total"] = sum(trx_agg.get(f"unique_dst_{m}", 0) for m in available_months)
    
    # Decay features (count-based)
    if "out_count_jan" in trx_agg.columns and "out_count_feb" in trx_agg.columns:
        trx_agg["trx_count_decay_jan_feb"] = (
            (trx_agg["out_count_feb"] - trx_agg["out_count_jan"]) / (trx_agg["out_count_jan"] + 1)
        )
    if "out_count_feb" in trx_agg.columns and "out_count_march" in trx_agg.columns:
        trx_agg["trx_count_decay_feb_march"] = (
            (trx_agg["out_count_march"] - trx_agg["out_count_feb"]) / (trx_agg["out_count_feb"] + 1)
        )
    if "out_count_jan" in trx_agg.columns and "out_count_march" in trx_agg.columns:
        trx_agg["trx_count_decay_jan_march"] = (
            (trx_agg["out_count_march"] - trx_agg["out_count_jan"]) / (trx_agg["out_count_jan"] + 1)
        )
    
    # === NEW: Per-type SPEND decay (feb → march) ===
    trx_types = ["P2P", "MerchantPay", "BillPay", "CashIn", "CashOut"]
    for ttype in trx_types:
        feb_col = f"sum_{ttype}_feb"
        mar_col = f"sum_{ttype}_march"
        if feb_col in trx_agg.columns and mar_col in trx_agg.columns:
            trx_agg[f"{ttype.lower()}_spend_decay_feb_march"] = (
                (trx_agg[mar_col] - trx_agg[feb_col]) / (trx_agg[feb_col] + 1)
            )
        # Per-type count decay
        feb_cnt = f"count_{ttype}_feb"
        mar_cnt = f"count_{ttype}_march"
        if feb_cnt in trx_agg.columns and mar_cnt in trx_agg.columns:
            trx_agg[f"{ttype.lower()}_count_decay_feb_march"] = (
                (trx_agg[mar_cnt] - trx_agg[feb_cnt]) / (trx_agg[feb_cnt] + 1)
            )
    
    # === NEW: Network diversity decay ===
    if "unique_dst_feb" in trx_agg.columns and "unique_dst_march" in trx_agg.columns:
        trx_agg["unique_dst_decay_feb_march"] = (
            (trx_agg["unique_dst_march"] - trx_agg["unique_dst_feb"]) / (trx_agg["unique_dst_feb"] + 1)
        )
    
    # === NEW: Active days decay ===
    if "active_days_feb" in trx_agg.columns and "active_days_march" in trx_agg.columns:
        trx_agg["active_days_decay_feb_march"] = (
            (trx_agg["active_days_march"] - trx_agg["active_days_feb"]) / (trx_agg["active_days_feb"] + 1)
        )
    
    # March activity share
    if "out_count_march" in trx_agg.columns:
        trx_agg["march_activity_share"] = trx_agg["out_count_march"] / (trx_agg["out_count_total"] + 1e-5)
    
    # Service diversity
    for ttype in trx_types:
        total = 0
        for m in available_months:
            col = f"count_{ttype}_{m}"
            if col in trx_agg.columns:
                total = total + trx_agg[col]
        trx_agg[f"count_{ttype}_total"] = total
    
    # Per-type sum totals
    for ttype in trx_types:
        total = 0
        for m in available_months:
            col = f"sum_{ttype}_{m}"
            if col in trx_agg.columns:
                total = total + trx_agg[col]
        trx_agg[f"sum_{ttype}_total"] = total
    
    total_type_cols = [f"count_{ttype}_total" for ttype in trx_types]
    trx_agg["trx_type_diversity"] = (trx_agg[total_type_cols] > 0).sum(axis=1)
    
    # Ratios
    denom = trx_agg["out_count_total"] + 1e-5
    trx_agg["merchant_pay_ratio"] = trx_agg["count_MerchantPay_total"] / denom
    trx_agg["bill_pay_ratio"] = trx_agg["count_BillPay_total"] / denom
    trx_agg["p2p_ratio"] = trx_agg["count_P2P_total"] / denom
    trx_agg["cashout_ratio"] = trx_agg["count_CashOut_total"] / denom
    
    # Net flow March
    if "in_sum_march" in trx_agg.columns and "out_sum_march" in trx_agg.columns:
        trx_agg["net_flow_march"] = trx_agg["in_sum_march"] - trx_agg["out_sum_march"]
    
    return trx_agg


def aggregate_balances(target_ids):
    """Aggregate daily balances — one month at a time, chunked pivot for March with advanced trend features."""
    print("Aggregating daily balances...")
    bal_files = sorted(glob.glob(os.path.join(base_path, "dayend_balance", "*.parquet")))
    months = ["jan", "feb", "march"]
    
    bal_agg = pd.DataFrame(index=list(target_ids))
    bal_agg.index.name = "ACCOUNT_ID"
    
    for month_idx, filepath in enumerate(bal_files):
        if month_idx >= len(months):
            break
        month = months[month_idx]
        print(f"  Processing balances for {month}...")
        
        df = pd.read_parquet(filepath, columns=["ACCOUNT_ID", "DATE", "AVAILABLE_BALANCE"])
        df = df[df["ACCOUNT_ID"].isin(target_ids)]
        
        # Monthly base stats
        stats = df.groupby("ACCOUNT_ID")["AVAILABLE_BALANCE"].agg(
            mean_bal="mean",
            std_bal="std",
            min_bal="min",
            max_bal="max"
        )
        stats.columns = [f"{col}_{month}" for col in stats.columns]
        bal_agg = bal_agg.join(stats, how="left").fillna(0)
        del stats
        
        # March detailed features — CHUNKED pivot to save memory
        if month == "march":
            df["day"] = pd.to_datetime(df["DATE"]).dt.day
            
            # Process pivot in chunks of 100K customers
            target_list = list(target_ids)
            chunk_size = 100_000
            
            final_bal = pd.Series(0.0, index=target_list, name="final_balance_march")
            bal_drop = pd.Series(0.0, index=target_list, name="balance_drop_march")
            bal_trend = pd.Series(0.0, index=target_list, name="balance_trend_march")
            zero_days = pd.Series(0, index=target_list, name="zero_balance_days_march")
            
            # New Advanced micro-window balance stats
            mean_bal_7d = pd.Series(0.0, index=target_list, name="mean_balance_last_7d_march")
            mean_bal_14d = pd.Series(0.0, index=target_list, name="mean_balance_last_14d_march")
            zero_days_7d = pd.Series(0, index=target_list, name="zero_balance_days_last_7d_march")
            bal_trend_14d = pd.Series(0.0, index=target_list, name="balance_trend_last_14d_march")
            
            for chunk_start in range(0, len(target_list), chunk_size):
                chunk_ids = set(target_list[chunk_start:chunk_start + chunk_size])
                chunk_df = df[df["ACCOUNT_ID"].isin(chunk_ids)]
                
                if len(chunk_df) == 0:
                    continue
                
                pivot = chunk_df.pivot(
                    index="ACCOUNT_ID", columns="day", values="AVAILABLE_BALANCE"
                ).fillna(0)
                
                # Align days to 1 to 31
                all_days = list(range(1, 32))
                pivot = pivot.reindex(columns=all_days, fill_value=0.0)
                
                # Final balance
                final_bal.update(pivot[31])
                
                # Balance drop
                bal_drop.update(pivot[31] - pivot[1])
                
                # Linear trend slope (31 days)
                t_centered = np.array(all_days) - 16.0
                denom_val = (t_centered ** 2).sum()
                weights = t_centered / denom_val
                trend = pd.Series(np.dot(pivot.values, weights), index=pivot.index)
                bal_trend.update(trend)
                
                # Zero balance days (31 days)
                zd = (pivot < 10.0).sum(axis=1)
                zero_days.update(zd)
                
                # Mean balance last 7 days (days 25-31)
                mean_7 = pivot[list(range(25, 32))].mean(axis=1)
                mean_bal_7d.update(mean_7)
                
                # Mean balance last 14 days (days 18-31)
                mean_14 = pivot[list(range(18, 32))].mean(axis=1)
                mean_bal_14d.update(mean_14)
                
                # Zero balance days last 7 days (days 25-31)
                zd_7 = (pivot[list(range(25, 32))] < 10.0).sum(axis=1)
                zero_days_7d.update(zd_7)
                
                # Linear trend last 14 days (days 18-31)
                t_14 = np.array(range(1, 15)) - 7.5
                denom_14 = (t_14 ** 2).sum()
                weights_14 = t_14 / denom_14
                trend_14 = pd.Series(np.dot(pivot[list(range(18, 32))].values, weights_14), index=pivot.index)
                bal_trend_14d.update(trend_14)
                
                del pivot, chunk_df
                gc.collect()
            
            bal_agg["final_balance_march"] = final_bal
            bal_agg["balance_drop_march"] = bal_drop
            bal_agg["balance_trend_march"] = bal_trend
            bal_agg["zero_balance_days_march"] = zero_days
            bal_agg["mean_balance_last_7d_march"] = mean_bal_7d
            bal_agg["mean_balance_last_14d_march"] = mean_bal_14d
            bal_agg["zero_balance_days_last_7d_march"] = zero_days_7d
            bal_agg["balance_trend_last_14d_march"] = bal_trend_14d
            
            del final_bal, bal_drop, bal_trend, zero_days, mean_bal_7d, mean_bal_14d, zero_days_7d, bal_trend_14d
        
        del df
        gc.collect()
    
    # Stability
    bal_agg["balance_stability_march"] = bal_agg["std_bal_march"] / (bal_agg["mean_bal_march"] + 1e-5)
    bal_agg["balance_change_jan_march"] = bal_agg["mean_bal_march"] - bal_agg["mean_bal_jan"]
    bal_agg["balance_change_feb_march"] = bal_agg["mean_bal_march"] - bal_agg["mean_bal_feb"]
    
    # New Final balance to mean balance ratio
    bal_agg["final_to_mean_balance_ratio_march"] = bal_agg["final_balance_march"] / (bal_agg["mean_bal_march"] + 1e-5)
    
    return bal_agg


def build_features(sample_fraction=1.0, seed=42):
    """Run end-to-end feature engineering with memory-safe processing and advanced features."""
    print(f"Starting feature engineering pipeline (sample={sample_fraction}, seed={seed})...")
    
    train_labels = pd.read_csv(os.path.join(base_path, "train_labels.csv"))
    test_labels = pd.read_csv(os.path.join(base_path, "test.csv"))
    
    if sample_fraction < 1.0:
        print(f"Sampling {sample_fraction*100}% of the data...")
        sampled_indices = train_labels.groupby("CHURN", group_keys=False).apply(
            lambda x: x.sample(frac=sample_fraction, random_state=seed)
        ).index
        train_sampled = train_labels.loc[sampled_indices].reset_index(drop=True)
        test_sampled = test_labels.sample(frac=sample_fraction, random_state=seed)
    else:
        train_sampled = train_labels
        test_sampled = test_labels
    
    target_ids = set(train_sampled["ACCOUNT_ID"]).union(set(test_sampled["ACCOUNT_ID"]))
    print(f"Total target customers: {len(target_ids):,}")
    
    # 1. KYC
    kyc_df = load_kyc(target_ids)
    gc.collect()
    
    # === NEW: Load KYC type map for graph/network features ===
    print("Loading KYC type map for network features...")
    kyc_full = pd.read_parquet(os.path.join(base_path, "kyc.parquet"), columns=["ACCOUNT_ID", "ACCOUNT_TYPE"])
    kyc_type_map = kyc_full.set_index("ACCOUNT_ID")["ACCOUNT_TYPE"].to_dict()
    del kyc_full
    gc.collect()
    
    # 2. Recency (lightweight pass across all 3 months)
    recency_df = compute_recency(target_ids)
    gc.collect()
    
    # 3. Advanced March transaction features
    march_adv_df = compute_advanced_march_features(target_ids)
    gc.collect()
    
    # 4. General transaction aggregations (Jan + Feb + March sums/counts + graph features)
    trx_df = aggregate_transactions(target_ids, kyc_type_map)
    del kyc_type_map
    trx_df = trx_df.join(recency_df, how="left")
    trx_df["days_since_last_trx"] = trx_df["days_since_last_trx"].fillna(90)
    del recency_df
    
    # Join advanced March transaction features
    trx_df = trx_df.join(march_adv_df, how="left")
    del march_adv_df
    gc.collect()
    
    # 5. Balance aggregation
    bal_df = aggregate_balances(target_ids)
    gc.collect()
    
    # 6. Join all
    features = kyc_df.join(trx_df, how="left").join(bal_df, how="left").fillna(0)
    del kyc_df, trx_df, bal_df
    gc.collect()
    
    # 7. Sparsity flags
    features["flag_zero_bill_pay"] = (features["bill_pay_ratio"] == 0).astype(float)
    features["flag_zero_merchant_pay"] = (features["merchant_pay_ratio"] == 0).astype(float)
    if "out_count_march" in features.columns:
        features["flag_zero_march_trx"] = (features["out_count_march"] == 0).astype(float)
        
    # Zero activity in the last 7 days of March flag
    if "out_count_last_7d" in features.columns:
        features["flag_zero_trx_last_7d_march"] = (features["out_count_last_7d"] == 0).astype(float)
    
    # === NEW: Balance-Transaction Interaction Features ===
    print("Computing balance-transaction interaction features...")
    if "mean_bal_march" in features.columns and "out_sum_march" in features.columns:
        features["balance_to_spend_ratio_march"] = features["mean_bal_march"] / (features["out_sum_march"] + 1)
        features["trx_intensity_march"] = features["out_count_march"] / (features["mean_bal_march"] + 1)
        features["balance_per_trx_march"] = features["mean_bal_march"] / (features["out_count_march"] + 1)
    
    if "mean_bal_feb" in features.columns and "out_sum_feb" in features.columns:
        features["balance_to_spend_ratio_feb"] = features["mean_bal_feb"] / (features["out_sum_feb"] + 1)
    
    # Balance runway: final balance / average daily spend
    if "final_balance_march" in features.columns and "out_sum_march" in features.columns:
        features["balance_runway_march"] = features["final_balance_march"] / (features["out_sum_march"] / 31 + 1)
    
    # 8. Log transforms for skewed monetary columns
    skewed_cols = [
        "out_sum_total", "in_sum_total", "out_sum_march", "in_sum_march",
        "mean_bal_march", "std_bal_march", "final_balance_march",
        "out_sum_last_7d", "out_sum_last_14d", "in_sum_last_7d", "in_sum_last_14d",
        "mean_balance_last_7d_march", "mean_balance_last_14d_march",
        "out_sum_jan", "out_sum_feb", "in_sum_jan", "in_sum_feb",
        "sum_MerchantPay_total", "sum_BillPay_total", "sum_P2P_total",
        "balance_to_spend_ratio_march", "balance_runway_march"
    ]
    for col in skewed_cols:
        if col in features.columns:
            features[f"{col}_log"] = np.log1p(features[col].clip(lower=0))
    
    # 9. Split back
    features = features.reset_index()
    
    train_features = features[features["ACCOUNT_ID"].isin(train_sampled["ACCOUNT_ID"])].copy()
    train_features = train_features.merge(train_sampled, on="ACCOUNT_ID", how="left")
    
    test_features = features[features["ACCOUNT_ID"].isin(test_sampled["ACCOUNT_ID"])].copy()
    
    del features
    gc.collect()
    
    print(f"\nTrain features: {train_features.shape}, Test features: {test_features.shape}")
    
    os.makedirs("./processed_data", exist_ok=True)
    train_features.to_parquet("./processed_data/train_features.parquet")
    test_features.to_parquet("./processed_data/test_features.parquet")
    print("Features saved to processed_data/.")

### ▶️ Run Feature Engineering

In [ ]:
start_time = time.time()
build_features(sample_fraction=1.0)
elapsed = time.time() - start_time
print(f"\n⏱️ Feature engineering completed in {elapsed/60:.1f} minutes")

## 🤖 Step 5: Model Zoo Training

Trains 5-6 diverse base classifiers with **10-Fold Stratified CV**, early stopping, and GPU auto-detection:
- **LightGBM** (3000 trees, regularized)
- **XGBoost** (3000 trees, regularized)
- **CatBoost** (3000 iterations, if installed)
- **Random Forest** (500 trees)
- **Logistic Regression** (scaled features)
- **MLP Neural Network** (3-layer, scaled features)

In [ ]:
# N_SPLITS and RANDOM_STATE already set in Step 3

def load_data():
    """Load the engineered features."""
    train_df = pd.read_parquet("./processed_data/train_features.parquet")
    test_df = pd.read_parquet("./processed_data/test_features.parquet")
    return train_df, test_df


def prepare_inputs(train_df, test_df):
    """Separate features and target, and drop unneeded ID columns."""
    exclude_cols = ["ACCOUNT_ID", "CHURN"]
    feature_cols = [col for col in train_df.columns if col not in exclude_cols]
    
    X = train_df[feature_cols].copy()
    y = train_df["CHURN"].copy()
    X_test = test_df[feature_cols].copy()
    
    # Ensure all column names are strings and have no special characters
    clean_cols = [str(col).replace("[", "").replace("]", "").replace("<", "").replace(" ", "_") for col in X.columns]
    X.columns = clean_cols
    X_test.columns = clean_cols
    
    print(f"Features count: {len(X.columns)}")
    return X, y, X_test, X.columns.tolist()


def evaluate_predictions(y_true, y_pred_proba, threshold=0.5):
    """Compute standard classification metrics."""
    y_pred = (y_pred_proba >= threshold).astype(int)
    auc = roc_auc_score(y_true, y_pred_proba)
    brier = brier_score_loss(y_true, y_pred_proba)
    f1 = f1_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    return {
        "AUC": auc,
        "Brier": brier,
        "F1": f1,
        "Recall": recall,
        "Precision": precision
    }


def get_xgb_device_params():
    """Detect if GPU is available and return appropriate XGBoost parameters."""
    try:
        import torch
        if not torch.cuda.is_available():
            print("  [CPU] PyTorch detected no CUDA GPU. Using CPU for XGBoost.")
            return {"n_jobs": -1, "tree_method": "hist"}
        else:
            print("  [GPU] CUDA detected. Enabling GPU for XGBoost.")
            return {"device": "cuda", "tree_method": "hist"}
    except ImportError:
        pass
    
    try:
        # Dummy classifier test
        clf = xgb.XGBClassifier(device="cuda", n_estimators=1)
        clf.fit(np.random.rand(10, 2), np.random.randint(0, 2, 10))
        print("  [GPU] XGBoost CUDA training test passed. Enabling GPU.")
        return {"device": "cuda", "tree_method": "hist"}
    except Exception:
        pass
        
    try:
        clf = xgb.XGBClassifier(tree_method="gpu_hist", n_estimators=1)
        clf.fit(np.random.rand(10, 2), np.random.randint(0, 2, 10))
        print("  [GPU] XGBoost gpu_hist test passed. Enabling GPU.")
        return {"tree_method": "gpu_hist"}
    except Exception:
        pass
        
    print("  [CPU] No GPU detected for XGBoost. Using CPU.")
    return {"n_jobs": -1, "tree_method": "hist"}


def get_lgb_device_params():
    """Detect if GPU is available and return appropriate LightGBM parameters."""
    try:
        import torch
        if not torch.cuda.is_available():
            print("  [CPU] PyTorch detected no CUDA GPU. Using CPU for LightGBM.")
            return {"n_jobs": -1}
    except ImportError:
        pass

    try:
        model = lgb.LGBMClassifier(device="gpu", n_estimators=1)
        model.fit(np.random.rand(10, 2), np.random.randint(0, 2, 10))
        print("  [GPU] LightGBM GPU training test passed. Enabling GPU.")
        return {"device": "gpu"}
    except Exception:
        pass
        
    try:
        model = lgb.LGBMClassifier(device="cuda", n_estimators=1)
        model.fit(np.random.rand(10, 2), np.random.randint(0, 2, 10))
        print("  [GPU] LightGBM CUDA training test passed. Enabling GPU.")
        return {"device": "cuda"}
    except Exception:
        pass
        
    print("  [CPU] No GPU detected for LightGBM. Using CPU.")
    return {"n_jobs": -1}


def get_catboost_device_params():
    """Detect if GPU is available and return appropriate CatBoost parameters."""
    if not CATBOOST_AVAILABLE:
        return {}
    try:
        import torch
        if not torch.cuda.is_available():
            print("  [CPU] PyTorch detected no CUDA GPU. Using CPU for CatBoost.")
            return {"thread_count": -1}
    except ImportError:
        pass

    try:
        model = CatBoostClassifier(task_type="GPU", iterations=1)
        model.fit(np.random.rand(10, 2), np.random.randint(0, 2, 10), verbose=False)
        print("  [GPU] CatBoost GPU training test passed. Enabling GPU.")
        return {"task_type": "GPU"}
    except Exception:
        pass
        
    print("  [CPU] No GPU detected for CatBoost. Using CPU.")
    return {"thread_count": -1}


def train_lgb(X, y, X_test, cv, class_weight=None):
    """Train LightGBM with Stratified CV (deeper search space & GPU support)."""
    print("\nTraining LightGBM...")
    oof_preds = np.zeros(len(X))
    test_preds = np.zeros(len(X_test))
    models = []
    
    scale_pos_weight = None
    if class_weight == "balanced":
        scale_pos_weight = (len(y) - y.sum()) / y.sum()
        print(f"  Using scale_pos_weight: {scale_pos_weight:.4f}")
        
    gpu_params = get_lgb_device_params()
    
    for fold, (train_idx, val_idx) in enumerate(cv.split(X, y)):
        X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
        X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
        
        # Tuned tree parameters with regularization
        params = {
            "objective": "binary",
            "metric": "auc",
            "boosting_type": "gbdt",
            "n_estimators": 3000,
            "learning_rate": 0.02,
            "num_leaves": 127,
            "max_depth": 9,
            "min_child_samples": 20,
            "min_child_weight": 0.01,
            "subsample": 0.75,
            "subsample_freq": 1,
            "colsample_bytree": 0.7,
            "reg_alpha": 0.1,
            "reg_lambda": 1.0,
            "random_state": RANDOM_STATE + fold,
            "verbose": -1
        }
        params.update(gpu_params)
        
        if scale_pos_weight:
            params["scale_pos_weight"] = scale_pos_weight
            
        model = lgb.LGBMClassifier(**params)
        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            callbacks=[lgb.early_stopping(100, verbose=False)]
        )
        
        val_preds = model.predict_proba(X_val)[:, 1]
        oof_preds[val_idx] = val_preds
        test_preds += model.predict_proba(X_test)[:, 1] / cv.n_splits
        models.append(model)
        
    metrics = evaluate_predictions(y, oof_preds)
    print(f"LightGBM CV Results: AUC={metrics['AUC']:.5f}, Brier={metrics['Brier']:.5f}")
    return oof_preds, test_preds, models


def train_xgb(X, y, X_test, cv, class_weight=None):
    """Train XGBoost with Stratified CV (deeper search space & GPU support)."""
    print("\nTraining XGBoost...")
    oof_preds = np.zeros(len(X))
    test_preds = np.zeros(len(X_test))
    models = []
    
    scale_pos_weight = None
    if class_weight == "balanced":
        scale_pos_weight = (len(y) - y.sum()) / y.sum()
        print(f"  Using scale_pos_weight: {scale_pos_weight:.4f}")
        
    gpu_params = get_xgb_device_params()
    
    for fold, (train_idx, val_idx) in enumerate(cv.split(X, y)):
        X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
        X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
        
        params = {
            "objective": "binary:logistic",
            "eval_metric": "auc",
            "n_estimators": 3000,
            "learning_rate": 0.02,
            "max_depth": 8,
            "subsample": 0.75,
            "colsample_bytree": 0.7,
            "min_child_weight": 3,
            "gamma": 0.1,
            "reg_alpha": 0.1,
            "reg_lambda": 1.0,
            "max_delta_step": 1,
            "random_state": RANDOM_STATE + fold,
            "early_stopping_rounds": 150
        }
        params.update(gpu_params)
        
        if scale_pos_weight:
            params["scale_pos_weight"] = scale_pos_weight
            
        model = xgb.XGBClassifier(**params)
        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            verbose=False
        )
        
        val_preds = model.predict_proba(X_val)[:, 1]
        oof_preds[val_idx] = val_preds
        test_preds += model.predict_proba(X_test)[:, 1] / cv.n_splits
        models.append(model)
        
    metrics = evaluate_predictions(y, oof_preds)
    print(f"XGBoost CV Results: AUC={metrics['AUC']:.5f}, Brier={metrics['Brier']:.5f}")
    return oof_preds, test_preds, models


def train_catboost(X, y, X_test, cv, class_weight=None):
    """Train CatBoost with Stratified CV (GPU-enabled, high performance)."""
    if not CATBOOST_AVAILABLE:
        return None, None, None
        
    print("\nTraining CatBoost...")
    oof_preds = np.zeros(len(X))
    test_preds = np.zeros(len(X_test))
    models = []
    
    auto_class_weights = None
    if class_weight == "balanced":
        auto_class_weights = "Balanced"
        print("  Using auto_class_weights: Balanced")
        
    gpu_params = get_catboost_device_params()
    
    for fold, (train_idx, val_idx) in enumerate(cv.split(X, y)):
        X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
        X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
        
        params = {
            "iterations": 3000,
            "learning_rate": 0.02,
            "depth": 8,
            "eval_metric": "AUC",
            "random_seed": RANDOM_STATE + fold,
            "l2_leaf_reg": 3,
            "bagging_temperature": 0.8,
            "early_stopping_rounds": 150
        }
        params.update(gpu_params)
        
        if auto_class_weights:
            params["auto_class_weights"] = auto_class_weights
            
        model = CatBoostClassifier(**params)
        model.fit(
            X_train, y_train,
            eval_set=(X_val, y_val),
            use_best_model=True,
            verbose=False
        )
        
        val_preds = model.predict_proba(X_val)[:, 1]
        oof_preds[val_idx] = val_preds
        test_preds += model.predict_proba(X_test)[:, 1] / cv.n_splits
        models.append(model)
        
    metrics = evaluate_predictions(y, oof_preds)
    print(f"CatBoost CV Results: AUC={metrics['AUC']:.5f}, Brier={metrics['Brier']:.5f}")
    return oof_preds, test_preds, models


def train_rf(X, y, X_test, cv):
    """Train Random Forest with Stratified CV."""
    print("\nTraining Random Forest...")
    oof_preds = np.zeros(len(X))
    test_preds = np.zeros(len(X_test))
    models = []
    
    for fold, (train_idx, val_idx) in enumerate(cv.split(X, y)):
        X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
        X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
        
        # Deeper forest with regularization
        model = RandomForestClassifier(
            n_estimators=500,
            max_depth=15,
            min_samples_leaf=10,
            max_features="sqrt",
            random_state=RANDOM_STATE + fold,
            n_jobs=-1,
            class_weight="balanced"
        )
        model.fit(X_train, y_train)
        
        val_preds = model.predict_proba(X_val)[:, 1]
        oof_preds[val_idx] = val_preds
        test_preds += model.predict_proba(X_test)[:, 1] / cv.n_splits
        models.append(model)
        
    metrics = evaluate_predictions(y, oof_preds)
    print(f"Random Forest CV Results: AUC={metrics['AUC']:.5f}, Brier={metrics['Brier']:.5f}")
    return oof_preds, test_preds, models


def train_linear_models(X, y, X_test, cv):
    """Train Logistic Regression and MLP Classifier (requiring scaling)."""
    print("\nPreparing scaled data for linear/neural models...")
    
    # Simple Imputer and Scaler Pipeline
    imputer = SimpleImputer(strategy="median")
    scaler = StandardScaler()
    
    # Scale inputs
    X_imputed = imputer.fit_transform(X)
    X_test_imputed = imputer.transform(X_test)
    
    X_scaled = pd.DataFrame(scaler.fit_transform(X_imputed), columns=X.columns)
    X_test_scaled = pd.DataFrame(scaler.transform(X_test_imputed), columns=X_test.columns)
    
    # 1. Logistic Regression
    print("Training Logistic Regression...")
    lr_oof = np.zeros(len(X))
    lr_test = np.zeros(len(X_test))
    lr_models = []
    
    for fold, (train_idx, val_idx) in enumerate(cv.split(X_scaled, y)):
        X_train, y_train = X_scaled.iloc[train_idx], y.iloc[train_idx]
        X_val, y_val = X_scaled.iloc[val_idx], y.iloc[val_idx]
        
        model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE + fold, class_weight="balanced")
        model.fit(X_train, y_train)
        
        lr_oof[val_idx] = model.predict_proba(X_val)[:, 1]
        lr_test += model.predict_proba(X_test_scaled)[:, 1] / cv.n_splits
        lr_models.append(model)
        
    lr_metrics = evaluate_predictions(y, lr_oof)
    print(f"Logistic Regression CV Results: AUC={lr_metrics['AUC']:.5f}, Brier={lr_metrics['Brier']:.5f}")
    
    # 2. MLP Classifier
    print("Training MLP (Neural Network) Classifier...")
    mlp_oof = np.zeros(len(X))
    mlp_test = np.zeros(len(X_test))
    mlp_models = []
    
    for fold, (train_idx, val_idx) in enumerate(cv.split(X_scaled, y)):
        X_train, y_train = X_scaled.iloc[train_idx], y.iloc[train_idx]
        X_val, y_val = X_scaled.iloc[val_idx], y.iloc[val_idx]
        
        # Deeper neural network for PC version
        model = MLPClassifier(
            hidden_layer_sizes=(256, 128, 64),
            activation="relu",
            max_iter=200,
            alpha=0.0005,
            batch_size=1024,
            random_state=RANDOM_STATE + fold,
            early_stopping=True
        )
        model.fit(X_train, y_train)
        
        mlp_oof[val_idx] = model.predict_proba(X_val)[:, 1]
        mlp_test += model.predict_proba(X_test_scaled)[:, 1] / cv.n_splits
        mlp_models.append(model)
        
    mlp_metrics = evaluate_predictions(y, mlp_oof)
    print(f"MLP CV Results: AUC={mlp_metrics['AUC']:.5f}, Brier={mlp_metrics['Brier']:.5f}")
    
    return lr_oof, lr_test, lr_models, mlp_oof, mlp_test, mlp_models


def plot_curves(y_true, oof_dict):
    """Plot AUC-ROC curves for all models in the zoo."""
    plt.figure(figsize=(10, 8))
    for model_name, oof_preds in oof_dict.items():
        if oof_preds is not None:
            fpr, tpr, _ = roc_curve(y_true, oof_preds)
            auc = roc_auc_score(y_true, oof_preds)
            plt.plot(fpr, tpr, label=f"{model_name} (AUC = {auc:.4f})")
        
    plt.plot([0, 1], [0, 1], "k--", label="Random Guess")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("Model Zoo ROC Curves (Out-of-Fold)")
    plt.legend(loc="lower right")
    plt.grid(True, alpha=0.3)
    
    os.makedirs("./plots", exist_ok=True)
    plt.savefig("./plots/roc_curves.png", dpi=300, bbox_inches="tight")
    print("ROC curve comparison plot saved to plots/roc_curves.png.")
    plt.close()


def main():
    train_df, test_df = load_data()
    X, y, X_test, feature_names = prepare_inputs(train_df, test_df)
    
    # Free the raw DataFrames to preserve RAM
    train_account_ids = train_df["ACCOUNT_ID"].copy()
    test_account_ids = test_df["ACCOUNT_ID"].copy()
    del train_df, test_df
    gc.collect()
    
    # N-Fold Stratified CV
    print(f"Using {N_SPLITS}-Fold Stratified Cross-Validation.")
    cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    
    # Train Model Zoo — sequential with memory cleanup between models
    print("\n=== Training Model Zoo ===")
    
    lgb_oof, lgb_test, lgb_models = train_lgb(X, y, X_test, cv, class_weight="balanced")
    gc.collect()
    
    xgb_oof, xgb_test, xgb_models = train_xgb(X, y, X_test, cv, class_weight="balanced")
    gc.collect()
    
    if CATBOOST_AVAILABLE:
        cat_oof, cat_test, cat_models = train_catboost(X, y, X_test, cv, class_weight="balanced")
        gc.collect()
    else:
        cat_oof, cat_test, cat_models = None, None, None
        
    rf_oof, rf_test, rf_models = train_rf(X, y, X_test, cv)
    gc.collect()
    
    lr_oof, lr_test, lr_models, mlp_oof, mlp_test, mlp_models = train_linear_models(X, y, X_test, cv)
    gc.collect()
    
    # Plot and save curves
    oof_dict = {
        "LightGBM": lgb_oof,
        "XGBoost": xgb_oof,
        "Random Forest": rf_oof,
        "Logistic Regression": lr_oof,
        "MLP Classifier": mlp_oof
    }
    if CATBOOST_AVAILABLE:
        oof_dict["CatBoost"] = cat_oof
        
    plot_curves(y, oof_dict)
    
    # Save Out-of-fold and test predictions for stacking meta-model
    os.makedirs("./predictions", exist_ok=True)
    
    oof_payload = {
        "ACCOUNT_ID": train_account_ids,
        "CHURN": y,
        "lgb_oof": lgb_oof,
        "xgb_oof": xgb_oof,
        "rf_oof": rf_oof,
        "lr_oof": lr_oof,
        "mlp_oof": mlp_oof
    }
    if CATBOOST_AVAILABLE:
        oof_payload["cat_oof"] = cat_oof
        
    oof_df = pd.DataFrame(oof_payload)
    oof_df.to_parquet("./predictions/oof_predictions.parquet")
    
    test_payload = {
        "ACCOUNT_ID": test_account_ids,
        "lgb_test": lgb_test,
        "xgb_test": xgb_test,
        "rf_test": rf_test,
        "lr_test": lr_test,
        "mlp_test": mlp_test
    }
    if CATBOOST_AVAILABLE:
        test_payload["cat_test"] = cat_test
        
    test_preds_df = pd.DataFrame(test_payload)
    test_preds_df.to_parquet("./predictions/test_predictions.parquet")
    
    os.makedirs("./models", exist_ok=True)
    joblib.dump(lgb_models[0], "./models/lgb_model.pkl")
    joblib.dump(xgb_models[0], "./models/xgb_model.pkl")
    if CATBOOST_AVAILABLE:
        joblib.dump(cat_models[0], "./models/cat_model.pkl")
        
    print("\nModel training phase completed successfully. All predictions saved.")

### ▶️ Run Model Training

In [ ]:
start_time = time.time()
main()  # This is the main() function from train.py
elapsed = time.time() - start_time
print(f"\n⏱️ Model training completed in {elapsed/60:.1f} minutes")

## 🏆 Step 6: Ensemble Optimization & Submission

Combines all base model predictions using:
1. **Stacking** (Logistic Regression meta-learner)
2. **AUC²-weighted Rank Average**
3. **Optuna-optimized Power-Rank Blending** (best method)

Then calibrates probabilities via Isotonic Regression and generates `predictions.csv`.

In [ ]:
def load_predictions():
    """Load OOF and test predictions from the model zoo."""
    oof = pd.read_parquet("./predictions/oof_predictions.parquet")
    test = pd.read_parquet("./predictions/test_predictions.parquet")
    return oof, test


def optimize_weights_optuna(oof_ranks, y, n_models, n_trials=200):
    """Use Optuna to find optimal blend weights maximizing OOF AUC."""
    print(f"  Running Optuna weight optimization ({n_trials} trials)...")
    
    def objective(trial):
        weights = np.array([trial.suggest_float(f"w{i}", 0.01, 1.0) for i in range(n_models)])
        weights = weights / weights.sum()
        blend = np.zeros(len(y))
        for i in range(n_models):
            blend += weights[i] * oof_ranks[i]
        return roc_auc_score(y, blend)
    
    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    
    best_weights = np.array([study.best_params[f"w{i}"] for i in range(n_models)])
    best_weights = best_weights / best_weights.sum()
    print(f"  Optuna best AUC: {study.best_value:.5f}")
    return best_weights


def optimize_weights_hillclimb(oof_ranks, y, n_models, n_iter=5000):
    """Hill climbing weight optimization as fallback."""
    print(f"  Running hill-climbing weight optimization ({n_iter} iterations)...")
    
    # Start from equal weights
    weights = np.ones(n_models) / n_models
    best_auc = 0
    best_weights = weights.copy()
    
    rng = np.random.RandomState(42)
    
    for i in range(n_iter):
        # Perturb weights
        new_weights = weights + rng.normal(0, 0.02, n_models)
        new_weights = np.clip(new_weights, 0.001, None)
        new_weights = new_weights / new_weights.sum()
        
        blend = np.zeros(len(y))
        for j in range(n_models):
            blend += new_weights[j] * oof_ranks[j]
        
        auc = roc_auc_score(y, blend)
        if auc > best_auc:
            best_auc = auc
            best_weights = new_weights.copy()
            weights = new_weights.copy()
    
    print(f"  Hill-climb best AUC: {best_auc:.5f}")
    return best_weights


def optimize_power_rank(oof_preds_list, y, stack_cols):
    """Find optimal power parameter for rank transformation (rank^power)."""
    print("  Optimizing rank power parameter...")
    
    best_auc = 0
    best_power = 1.0
    
    for power in np.arange(0.3, 3.0, 0.05):
        rank_sum = np.zeros(len(y))
        for preds in oof_preds_list:
            r = (rankdata(preds) / len(preds)) ** power
            rank_sum += r
        auc = roc_auc_score(y, rank_sum)
        if auc > best_auc:
            best_auc = auc
            best_power = power
    
    print(f"  Best rank power: {best_power:.2f} (AUC: {best_auc:.5f})")
    return best_power


def build_stacking_ensemble():
    """Train a Logistic Regression/Ridge meta-classifier on stacked OOF predictions and compare with Rank-Average Blending."""
    oof, test = load_predictions()
    
    # Identify available base models dynamically
    stack_cols = [c for c in oof.columns if c.endswith("_oof")]
    test_stack_cols = [c.replace("_oof", "_test") for c in stack_cols]
    
    print("Detected base models for ensembling:")
    for col in stack_cols:
        print(f"  - {col[:-4]}")
        
    X_stack = oof[stack_cols].values
    y = oof["CHURN"].values
    X_test_stack = test[test_stack_cols].values
    
    n_models = len(stack_cols)
    
    # Results dictionary
    results = {}
    
    # 1. Individual Model Metrics
    print("\n--- Individual Base Model AUCs ---")
    individual_aucs = {}
    for col in stack_cols:
        auc = roc_auc_score(y, oof[col].values)
        brier = brier_score_loss(y, oof[col].values)
        individual_aucs[col] = auc
        print(f"  {col[:-4]:<20}: AUC = {auc:.5f}, Brier = {brier:.5f}")
        results[f"Base: {col[:-4]}"] = {
            "auc": auc,
            "brier": brier,
            "oof": oof[col].values,
            "test": test[col.replace('_oof', '_test')].values
        }

    # 2. Stacking Meta-Classifier (Logistic Regression)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    meta_oof = np.zeros(len(X_stack))
    meta_test = np.zeros(len(X_test_stack))
    
    print("\nTraining Stacking Meta-Classifier (Logistic Regression)...")
    for fold, (train_idx, val_idx) in enumerate(cv.split(X_stack, y)):
        X_train, y_train = X_stack[train_idx], y[train_idx]
        X_val = X_stack[val_idx]
        
        meta_model = LogisticRegression(max_iter=1000, random_state=42 + fold)
        meta_model.fit(X_train, y_train)
        
        meta_oof[val_idx] = meta_model.predict_proba(X_val)[:, 1]
        meta_test += meta_model.predict_proba(X_test_stack)[:, 1] / cv.n_splits
    
    # Calibrate Stacking
    iso_meta = IsotonicRegression(out_of_bounds="clip")
    iso_meta.fit(meta_oof, y)
    calibrated_meta_oof = iso_meta.predict(meta_oof)
    calibrated_meta_test = iso_meta.predict(meta_test)
    
    results["Stacking (Calibrated LR)"] = {
        "auc": roc_auc_score(y, calibrated_meta_oof),
        "brier": brier_score_loss(y, calibrated_meta_oof),
        "oof": calibrated_meta_oof,
        "test": calibrated_meta_test
    }
    
    # 3. Simple Weighted Average Ensemble
    total_auc = sum(individual_aucs.values())
    weights = {col: auc / total_auc for col, auc in individual_aucs.items()}
    
    weighted_oof = np.zeros(len(X_stack))
    weighted_test = np.zeros(len(X_test_stack))
    for oof_col, test_col in zip(stack_cols, test_stack_cols):
        w = weights[oof_col]
        weighted_oof += w * oof[oof_col].values
        weighted_test += w * test[test_col].values
        
    results["Weighted Average"] = {
        "auc": roc_auc_score(y, weighted_oof),
        "brier": brier_score_loss(y, weighted_oof),
        "oof": weighted_oof,
        "test": weighted_test
    }

    # 4. Rank-Average Blending with AUC-squared weights (baseline)
    print("\nComputing Rank-Average Blend (AUC-squared weights)...")
    rank_oof_baseline = np.zeros(len(X_stack))
    rank_test_baseline = np.zeros(len(X_test_stack))
    
    for oof_col, test_col in zip(stack_cols, test_stack_cols):
        w = (individual_aucs[oof_col] ** 2) / sum(auc ** 2 for auc in individual_aucs.values())
        r_oof = rankdata(oof[oof_col].values) / len(oof)
        r_test = rankdata(test[test_col].values) / len(test)
        rank_oof_baseline += w * r_oof
        rank_test_baseline += w * r_test
        
    iso_rank_base = IsotonicRegression(out_of_bounds="clip")
    iso_rank_base.fit(rank_oof_baseline, y)
    
    results["Rank-Avg (AUC² weights)"] = {
        "auc": roc_auc_score(y, iso_rank_base.predict(rank_oof_baseline)),
        "brier": brier_score_loss(y, iso_rank_base.predict(rank_oof_baseline)),
        "oof": iso_rank_base.predict(rank_oof_baseline),
        "test": iso_rank_base.predict(rank_test_baseline)
    }

    # 5. === NEW: Optuna/Hill-Climbing Optimized Rank-Average Blend ===
    print("\n--- Optimized Rank-Average Blend ---")
    
    # Pre-compute OOF and test ranks
    oof_ranks = []
    test_ranks = []
    for oof_col, test_col in zip(stack_cols, test_stack_cols):
        oof_ranks.append(rankdata(oof[oof_col].values) / len(oof))
        test_ranks.append(rankdata(test[test_col].values) / len(test))
    
    # Find optimal power for rank transformation
    oof_preds_list = [oof[col].values for col in stack_cols]
    best_power = optimize_power_rank(oof_preds_list, y, stack_cols)
    
    # Apply power transform to ranks
    oof_ranks_power = [(rankdata(oof[col].values) / len(oof)) ** best_power for col in stack_cols]
    test_ranks_power = [(rankdata(test[col].values) / len(test)) ** best_power for col in test_stack_cols]
    
    # Optimize weights
    if OPTUNA_AVAILABLE:
        opt_weights = optimize_weights_optuna(oof_ranks_power, y, n_models, n_trials=300)
    else:
        opt_weights = optimize_weights_hillclimb(oof_ranks_power, y, n_models, n_iter=8000)
    
    print("  Optimized weights:")
    for col, w in zip(stack_cols, opt_weights):
        print(f"    {col[:-4]:<15}: {w:.4f}")
    
    # Apply optimized blend
    opt_rank_oof = np.zeros(len(X_stack))
    opt_rank_test = np.zeros(len(X_test_stack))
    for i, (oof_col, test_col) in enumerate(zip(stack_cols, test_stack_cols)):
        opt_rank_oof += opt_weights[i] * oof_ranks_power[i]
        opt_rank_test += opt_weights[i] * test_ranks_power[i]
    
    # Calibrate
    iso_opt = IsotonicRegression(out_of_bounds="clip")
    iso_opt.fit(opt_rank_oof, y)
    calibrated_opt_oof = iso_opt.predict(opt_rank_oof)
    calibrated_opt_test = iso_opt.predict(opt_rank_test)
    
    results["Optimized Power-Rank Blend"] = {
        "auc": roc_auc_score(y, calibrated_opt_oof),
        "brier": brier_score_loss(y, calibrated_opt_oof),
        "oof": calibrated_opt_oof,
        "test": calibrated_opt_test
    }
    
    # --- Cost-Sensitive Threshold Optimization on final best OOF predictions ---
    best_method = max(results, key=lambda k: results[k]["auc"])
    best_oof_proba = results[best_method]["oof"]
    
    print(f"\nBest ensemble method: {best_method} (AUC: {results[best_method]['auc']:.5f})")
    
    print("\n--- Cost-Sensitive Threshold Optimization ---")
    best_threshold = 0.5
    best_cost = float("inf")
    
    for threshold in np.arange(0.1, 0.9, 0.01):
        y_pred = (best_oof_proba >= threshold).astype(int)
        fn = ((y == 1) & (y_pred == 0)).sum()
        fp = ((y == 0) & (y_pred == 1)).sum()
        cost = 5 * fn + 1 * fp
        if cost < best_cost:
            best_cost = cost
            best_threshold = threshold
            
    print(f"Optimal threshold: {best_threshold:.2f} (Expected Loss: {best_cost})")
    
    # --- Generate Submission ---
    final_test_proba = results[best_method]["test"]
    
    submission = pd.DataFrame({
        "ACCOUNT_ID": test["ACCOUNT_ID"],
        "CHURN_PROB": final_test_proba
    })
    submission.to_csv("./predictions.csv", index=False)
    print(f"\nSubmission saved to predictions.csv with CHURN_PROB probabilities.")
    
    # Also save probabilities for SHAP and segment analysis
    proba_df = pd.DataFrame({
        "ACCOUNT_ID": test["ACCOUNT_ID"],
        "churn_probability": final_test_proba
    })
    os.makedirs("./predictions", exist_ok=True)
    proba_df.to_csv("./predictions/test_probabilities.csv", index=False)
    
    # Print comparison summary
    print("\n=== ENSEMBLE COMPARISON SUMMARY ===")
    print(f"{'Method':<35} {'AUC':>8} {'Brier':>8}")
    print("-" * 55)
    for method, vals in results.items():
        marker = " <<<" if method == best_method else ""
        print(f"{method:<35} {vals['auc']:>8.5f} {vals['brier']:>8.5f}{marker}")

### ▶️ Run Ensemble & Generate Submission

In [ ]:
start_time = time.time()
build_stacking_ensemble()
elapsed = time.time() - start_time
print(f"\n⏱️ Ensemble completed in {elapsed/60:.1f} minutes")

## 📥 Step 7: Download Submission File

In [ ]:
# Verify submission format
sub = pd.read_csv("./predictions.csv")
print(f"Submission shape: {sub.shape}")
print(f"Columns: {list(sub.columns)}")
print(f"CHURN_PROB range: [{sub['CHURN_PROB'].min():.6f}, {sub['CHURN_PROB'].max():.6f}]")
print(f"\nFirst 5 rows:")
print(sub.head())

# Also copy to Drive for safety
import shutil
drive_output = "/content/drive/MyDrive/predictions.csv"
shutil.copy("./predictions.csv", drive_output)
print(f"\n✅ Submission saved to Google Drive: {drive_output}")

# Download to local machine
from google.colab import files
files.download("./predictions.csv")

## ✅ Done!

Your `predictions.csv` has been:
1. Saved to the current Colab directory
2. Copied to your Google Drive root
3. Downloaded to your local machine

Upload it to the Kaggle competition page to see your score. 🎯